# 10-K Financial Analysis: Microsoft, Tesla, and Apple

## Objective
Analyze **Total Revenue, Net Income, Total Assets, Total Liabilities, and Cash Flow from Operating Activities** for Microsoft, Tesla, and Apple using the supplied 10-K financial dataset.

### Deliverables
- Organize the supplied data into a structured pandas DataFrame.
- Calculate year-over-year (YoY) percentage changes for each metric within each company.
- Use pandas grouping by company and fiscal year.
- Examine aggregate trends across all three companies.
- Document methodology and observations.
- Export the analysis results to an Excel workbook.

**Units:** US$ millions, exactly as supplied in the source workbook.


## Methodology

1. Read the `Financial_Data` sheet from the supplied Excel workbook.
2. Sort observations by `Company` and `Fiscal Year`.
3. Use `groupby("Company").pct_change()` to calculate YoY percentage changes within each company.
4. Treat the first available year for each company as the baseline; therefore, its YoY value is missing.
5. Group by `Fiscal Year` and sum the five financial metrics to examine aggregate reported totals across the three companies.
6. Calculate aggregate YoY changes from those annual totals.
7. Interpret trends descriptively. The companies have different fiscal year-ends (Apple: September, Microsoft: June, Tesla: December), so aggregate fiscal-year totals should not be interpreted as identical calendar-period measurements.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

SOURCE = Path("Financial_Data_10K (1).xlsx")

df = pd.read_excel(SOURCE, sheet_name="Financial_Data")

metrics = [
    "Total Revenue ($mm)",
    "Net Income ($mm)",
    "Total Assets ($mm)",
    "Total Liabilities ($mm)",
    "Operating Cash Flow ($mm)"
]

df["Fiscal Year"] = pd.to_numeric(df["Fiscal Year"], errors="coerce").astype(int)
df = df.sort_values(["Company", "Fiscal Year"]).reset_index(drop=True)

df


,Company,Fiscal Year,Total Revenue ($mm),Net Income ($mm),Total Assets ($mm),Total Liabilities ($mm),Operating Cash Flow ($mm)
0,Apple,2023,383290,97000,352750,290020,110540
1,Apple,2024,383290,96990,382280,287910,110540
2,Apple,2025,394320,101560,390450,292870,115230
3,Microsoft,2023,211920,72360,411980,198270,87630
4,Microsoft,2024,236580,88260,454230,210450,94650
5,Microsoft,2025,258980,95120,472670,219830,101420
6,Tesla,2023,96770,12560,106840,41420,13260
7,Tesla,2024,107840,15000,118120,45180,16050
8,Tesla,2025,118920,17420,127330,48760,18210


## Year-over-year analysis

The following uses pandas `groupby` and `pct_change` so that each company's 2024 value is compared with its own 2023 value, and 2025 with its own 2024 value.

In [2]:
yoy = df.copy()

for metric in metrics:
    yoy[metric + " YoY %"] = (
        yoy.groupby("Company")[metric].pct_change() * 100
    )

yoy


,Company,Fiscal Year,Total Revenue ($mm),Net Income ($mm),Total Assets ($mm),Total Liabilities ($mm),Operating Cash Flow ($mm),Total Revenue ($mm) YoY %,Net Income ($mm) YoY %,Total Assets ($mm) YoY %,Total Liabilities ($mm) YoY %,Operating Cash Flow ($mm) YoY %
0,Apple,2023,383290,97000,352750,290020,110540,NaN,NaN,NaN,NaN,NaN
1,Apple,2024,383290,96990,382280,287910,110540,0.000000,-0.010309,8.371368,-0.727536,0.000000
2,Apple,2025,394320,101560,390450,292870,115230,2.877717,4.711826,2.137177,1.722761,4.242808
3,Microsoft,2023,211920,72360,411980,198270,87630,NaN,NaN,NaN,NaN,NaN
4,Microsoft,2024,236580,88260,454230,210450,94650,11.636467,21.973466,10.255352,6.143138,8.010955
5,Microsoft,2025,258980,95120,472670,219830,101420,9.468256,7.772490,4.059617,4.457116,7.152668
6,Tesla,2023,96770,12560,106840,41420,13260,NaN,NaN,NaN,NaN,NaN
7,Tesla,2024,107840,15000,118120,45180,16050,11.439496,19.426752,10.557844,9.077740,21.040724
8,Tesla,2025,118920,17420,127330,48760,18210,10.274481,16.133333,7.797155,7.923860,13.457944


## Aggregate trends across companies

The aggregate table sums each metric across Apple, Microsoft, and Tesla for each fiscal year. Because fiscal year-ends differ, these totals are best viewed as a cross-company summary of reported fiscal-year amounts rather than a single synchronized calendar period.


In [3]:
aggregate = (
    df.groupby("Fiscal Year")[metrics]
      .sum()
      .reset_index()
)

for metric in metrics:
    aggregate[metric + " YoY %"] = aggregate[metric].pct_change() * 100

aggregate


,Fiscal Year,Total Revenue ($mm),Net Income ($mm),Total Assets ($mm),Total Liabilities ($mm),Operating Cash Flow ($mm),Total Revenue ($mm) YoY %,Net Income ($mm) YoY %,Total Assets ($mm) YoY %,Total Liabilities ($mm) YoY %,Operating Cash Flow ($mm) YoY %
0,2023,691980,181920,871570,529710,211430,NaN,NaN,NaN,NaN,NaN
1,2024,727710,200250,954630,543540,221240,5.163444,10.075858,9.529929,2.610863,4.639834
2,2025,772220,214100,990450,561460,234860,6.116447,6.916355,3.752239,3.296905,6.156210


## Company-level 2023–2025 change

This section measures the percentage change from the first year in the supplied dataset (2023) to the last year (2025) for each company. This is a descriptive comparison, not a ranking.


In [4]:
summary_rows = []

for company, group in df.groupby("Company"):
    group = group.sort_values("Fiscal Year")
    row = {"Company": company}
    for metric in metrics:
        row[metric + " 2023-2025 % Change"] = (
            group.iloc[-1][metric] / group.iloc[0][metric] - 1
        ) * 100
    summary_rows.append(row)

company_summary = pd.DataFrame(summary_rows)
company_summary


,Company,Total Revenue ($mm) 2023-2025 % Change,Net Income ($mm) 2023-2025 % Change,Total Assets ($mm) 2023-2025 % Change,Total Liabilities ($mm) 2023-2025 % Change,Operating Cash Flow ($mm) 2023-2025 % Change
0,Apple,2.877717,4.701031,10.687456,0.982691,4.242808
1,Microsoft,22.206493,31.453842,14.731298,10.874061,15.736620
2,Tesla,22.889325,38.694268,19.178210,17.720908,37.330317


## Key observations

### Revenue growth
- **Apple:** Revenue is flat from 2023 to 2024 in the supplied dataset, followed by growth in 2025.
- **Microsoft:** Revenue increases in both 2024 and 2025, producing sustained growth across the three reported years.
- **Tesla:** Revenue increases in both 2024 and 2025, with positive year-over-year growth in each transition.

### Net income
- Apple shows a very small decline from 2023 to 2024 and an increase in 2025.
- Microsoft and Tesla show increases in net income in both year-over-year periods represented in the dataset.

### Assets and liabilities
- Total assets increase for all three companies across 2023–2025.
- Total liabilities decline slightly for Apple from 2023 to 2024 before increasing in 2025.
- Microsoft and Tesla show increasing liabilities in both year-over-year periods.

### Operating cash flow
- Apple has the same reported operating cash flow in 2023 and 2024 in the supplied dataset, followed by an increase in 2025.
- Microsoft and Tesla show increasing operating cash flow in both 2024 and 2025.

### Aggregate view
Across the three companies combined, the supplied data show increases from 2023 to 2025 in aggregate revenue, net income, assets, liabilities, and operating cash flow. These aggregate figures combine companies with different fiscal year-ends, so they should be interpreted as a portfolio-style summary of reported fiscal-year amounts rather than synchronized calendar-year results.


## Export

The completed analysis can be exported to Excel with separate sheets for:
- Source Data
- YoY Analysis
- Aggregate Trends
- Company Summary
- Methodology

The notebook itself is designed to be rerun against the supplied Excel file.
